# e11 — The in-degree ladder (Axiomatics II)

`notes/in-degree-ladder.md` reported the discovery: over the exhaustive n = 3
universe, the deterministic maxima of φ_s(2023) conditional on the analyzed state's
in-degree m obey m·g(m) = n(n−1) for m ≤ 2, = n for 3 ≤ m ≤ 5, and 0 for m ≥ 6 —
two plateaus and a cliff, with the winners' partitioned-cause suppression saturating
at the severed capacity 2^(−n(n−1)) on the first plateau and at exactly 1/Q on the
second. This notebook takes the ladder to n = 4 (where the n = 3 coincidence
factory cannot confound), attacks the attainment-sufficiency gap, measures the
repaired measure's *stochastic* ceiling, and retries the weighted-attractor search
with ladder-informed seeds.

## Pre-registered predictions (written before execution)

- **P1 (the n = 4 ladder).** Search-estimated conditional maxima follow
  m·g(m) = n(n−1) = 12 for m ∈ {1, 2}; m·g(m) = n = 4 for 3 ≤ m ≤ Q−3 = 13; 0 for
  m ≥ Q−2 = 14. (The n = 3 cliff at 6 = Q−2 and second plateau n = log₂Q motivate
  the absolute-threshold-at-3 and cliff-at-Q−2 forms; search values are lower
  bounds, so the test is plateau *consistency*, not exact attainment.)
- **P2 (suppression mechanism).** The per-m winners' complete-cut partitioned
  cause likelihood equals 2^(−n(n−1)) on the first plateau and 1/Q on the second,
  at n = 4 as at n = 3.
- **P3 (sufficiency completion).** At n = 3, the capacity attainers are exactly
  the systems satisfying the necessary condition *plus its cause-side mirror at
  the preimage*: in-degree(0) = 1 with unique preimage u*, and for every unit j,
  output bit_j(0) is realized exactly once among the states agreeing with u* on
  unit j's self-input. Set equality against the 3,591, tested exactly.
- **P4 (the repaired measure's stochastic ceiling).** Ascending
  min(signed φ_2023, −log₂ p_c) over stochastic systems at n = 3 exceeds the
  deterministic ceiling 1.0 — registered: best ∈ (1, 3] — because without the ii
  term the cause crossing does not apply; anatomy will show which of MIP and
  cause surprisal binds.
- **P5 (weighted search, retried).** With ladder-informed seeding and heavier
  mutations, the n = 4 weighted-deterministic best exceeds 4.0 (the dead
  conjecture's value), moving toward the ladder bound n(n−1)/2 = 6.


In [1]:
import time
from pathlib import Path

import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np

import iitx
from iitx.measures import iit4
from iitx.system import System

DATA = Path("data")
SEED = 0
print(f"iitx {iitx.__version__}, jax {jax.__version__}, seed {SEED}")


def batch_phi_at_zero(n):
	state = jnp.zeros(n, dtype=jnp.int32)
	return jax.jit(jax.vmap(lambda t: iit4.system_phi(System.from_state_by_node(t), state).phi))


def indegree0(tables):
	"""Number of prior states mapping to the all-off state, per system."""
	return (tables.sum(axis=2) == 0).sum(axis=1)


def adam_step(params, grads, m, v, step, lr=0.02, b1=0.9, b2=0.999, eps=1e-8):
	m = b1 * m + (1 - b1) * grads
	v = b2 * v + (1 - b2) * grads**2
	m_hat = m / (1 - b1**step)
	v_hat = v / (1 - b2**step)
	return params + lr * m_hat / (jnp.sqrt(v_hat) + eps), m, v

iitx 0.1.0, jax 0.11.1, seed 0


## 1. The n = 4 ladder by conditioned search (P1)


In [2]:
N4, Q4 = 4, 16
phi4 = batch_phi_at_zero(N4)
rng = np.random.default_rng(SEED + 300)


def random_with_indegree(m, count):
	"""Random deterministic tables with exactly m preimages of the all-off state."""
	tables = (rng.random((count, Q4, N4)) > 0.5).astype(np.float64)
	for b in range(count):
		S = rng.choice(Q4, size=m, replace=False)
		tables[b, S, :] = 0.0
		outside = np.setdiff1d(np.arange(Q4), S)
		zero_rows = outside[tables[b, outside, :].sum(axis=1) == 0]
		for s in zero_rows:
			tables[b, s, rng.integers(0, N4)] = 1.0
	return tables


def ladder_rung(m, random_budget=16384, chains=128, generations=250):
	tables = random_with_indegree(m, random_budget)
	assert (indegree0(tables) == m).all()
	values = np.asarray(phi4(jnp.asarray(tables)))
	elite = tables[np.argsort(values)[-chains:]]
	elite_values = np.sort(values)[-chains:]
	for _ in range(generations):
		proposals = elite.copy()
		rows = np.arange(chains)
		s = rng.integers(0, Q4, size=chains)
		j = rng.integers(0, N4, size=chains)
		proposals[rows, s, j] = 1.0 - proposals[rows, s, j]
		ok = indegree0(proposals) == m
		proposal_values = np.where(ok, np.asarray(phi4(jnp.asarray(proposals))), -1.0)
		accept = proposal_values > elite_values
		elite[accept] = proposals[accept]
		elite_values = np.where(accept, proposal_values, elite_values)
	k = int(np.argmax(elite_values))
	return float(elite_values[k]), elite[k]


ladder = {}
winners = {}
start = time.perf_counter()
for m in (1, 2, 3, 4, 5, 6, 8, 10, 12, 14):
	ladder[m], winners[m] = ladder_rung(m)
	print(f"m={m:>2}: g~(m) = {ladder[m]:.4f}   m*g = {m * ladder[m]:.3f}")
print(f"({time.perf_counter() - start:,.0f} s; search values are lower bounds)")

m= 1: g~(m) = 4.2451   m*g = 4.245


m= 2: g~(m) = 2.1226   m*g = 4.245


m= 3: g~(m) = 1.4150   m*g = 4.245


m= 4: g~(m) = 0.7500   m*g = 3.000


m= 5: g~(m) = 0.8000   m*g = 4.000


m= 6: g~(m) = 0.5000   m*g = 3.000


m= 8: g~(m) = 0.2945   m*g = 2.356


m=10: g~(m) = 0.2034   m*g = 2.034


m=12: g~(m) = 0.1038   m*g = 1.245


m=14: g~(m) = 0.0000   m*g = 0.000
(116 s; search values are lower bounds)


## 2. The suppression mechanism at the winners (P2)

The complete cut severs all cross-connections; the partitioned likelihood of the
specified cause state factorizes over units with cross-inputs uniform and the
self-input at the cause state's value.


In [3]:
def complete_cut_suppression(table, n):
	"""Partitioned likelihood of reaching all-off from the (2023-)specified cause
	state under the complete cut, computed directly from the table."""
	q = 2**n
	state = jnp.zeros(n, dtype=jnp.int32)
	result = iit4.system_phi(System.from_state_by_node(jnp.asarray(table)), state)
	u = np.asarray(result.cause_effect_state.cause_state)
	value = 1.0
	for j in range(n):
		rows = [s for s in range(q) if ((s >> j) & 1) == u[j]]
		value *= np.mean(table[rows, j] == 0)
	return float(value), float(result.phi)


print(f"{'m':>3} {'g~':>8} {'suppression':>13} {'2^-n(n-1)':>11} {'1/Q':>8}")
for m in sorted(winners):
	if ladder[m] <= 0:
		continue
	supp, phi = complete_cut_suppression(winners[m], N4)
	print(f"{m:>3} {ladder[m]:>8.4f} {supp:>13.3e} {2.0**-12:>11.3e} {1 / Q4:>8.4f}")

  m       g~   suppression   2^-n(n-1)      1/Q


  1   4.2451     1.318e-02   2.441e-04   0.0625
  2   2.1226     1.318e-02   2.441e-04   0.0625
  3   1.4150     1.978e-02   2.441e-04   0.0625
  4   0.7500     4.688e-02   2.441e-04   0.0625
  5   0.8000     6.250e-02   2.441e-04   0.0625
  6   0.5000     6.250e-02   2.441e-04   0.0625


  8   0.2945     1.221e-01   2.441e-04   0.0625
 10   0.2034     1.221e-01   2.441e-04   0.0625
 12   0.1038     2.637e-01   2.441e-04   0.0625


## 3. Closing the sufficiency gap at n = 3 (P3)


In [4]:
phi23 = np.load(DATA / "e01_sweep.npy")
M3, N3, Q3 = 2**24, 3, 8
codes = np.arange(M3, dtype=np.uint32)
columns = np.stack([((codes >> (8 * i)) & 0xFF) for i in range(N3)], axis=1)
attainers = phi23 == 6.0

# Necessary condition (e09, corrected): unique preimage + cross-minterm effect.
indeg = np.zeros(M3, dtype=np.uint32)
preimage = np.zeros(M3, dtype=np.int64)
for u in range(Q3):
	to_zero = np.ones(M3, dtype=bool)
	for i in range(N3):
		to_zero &= ((columns[:, i] >> u) & 1) == 0
	indeg += to_zero
	preimage = np.where(to_zero, u, preimage)

effect_ok = np.ones(M3, dtype=bool)
for j in range(N3):
	out0 = columns[:, j] & 1
	count = np.zeros(M3, dtype=np.uint8)
	for s in range(Q3):
		if ((s >> j) & 1) == 0:
			count += ((columns[:, j] >> s) & 1) == out0
	effect_ok &= count == 1
necessary = effect_ok & (indeg == 1)

# Candidate completion: the cause-side mirror at the unique preimage u*: for each
# unit j, output 0 (= bit_j of the analyzed all-off state) is realized exactly once
# among prior states agreeing with u* on unit j's self-input.
cause_ok = np.ones(M3, dtype=bool)
for j in range(N3):
	count = np.zeros(M3, dtype=np.uint8)
	for s in range(Q3):
		agree = ((preimage >> j) & 1) == ((s >> j) & 1)
		outputs_zero = ((columns[:, j] >> s) & 1) == 0
		count += (agree & outputs_zero).astype(np.uint8)
	cause_ok &= count == 1
candidate = necessary & cause_ok

print(
	f"attainers {attainers.sum():,}; necessary {necessary.sum():,}; "
	f"necessary+mirror {candidate.sum():,}"
)
print(
	f"candidate XOR attainers: {(candidate ^ attainers).sum():,} "
	f"(candidate-only {(candidate & ~attainers).sum():,}, "
	f"attainer-only {(attainers & ~candidate).sum():,})"
)

attainers 3,591; necessary 13,679; necessary+mirror 3,591
candidate XOR attainers: 0 (candidate-only 0, attainer-only 0)


## 4. The repaired measure's stochastic ceiling (P4)


In [5]:
N3j, Q3j = 3, 8
STATE3 = jnp.zeros(N3j, dtype=jnp.int32)
BITS3 = jnp.asarray([[(v >> i) & 1 for i in range(N3j)] for v in range(Q3j)], dtype=jnp.float64)


def cause_capped_signed(logits):
	p = jax.nn.sigmoid(logits)
	result = iit4.system_phi(System.from_state_by_node(p), STATE3)
	column0 = jnp.prod(1.0 - p, axis=1)
	total = jnp.sum(column0)
	cause_code = jnp.sum(result.cause_effect_state.cause_state * (2 ** jnp.arange(N3j)))
	p_c = jnp.where(total > 0, column0[cause_code] / jnp.where(total > 0, total, 1.0), 1.0)
	cap = -jnp.log2(jnp.maximum(p_c, 1e-300))
	return jnp.minimum(result.signed_phi, cap)


value_and_grad = jax.jit(jax.vmap(jax.value_and_grad(cause_capped_signed)))
batch_value = jax.jit(jax.vmap(cause_capped_signed))
rng2 = np.random.default_rng(SEED + 400)
logits = jnp.asarray(0.5 * rng2.standard_normal((2048, Q3j, N3j)))
m, v = jnp.zeros_like(logits), jnp.zeros_like(logits)
start = time.perf_counter()
for step in range(1, 1501):
	lr = 0.02 if step <= 1000 else 0.002
	_, grads = value_and_grad(logits)
	logits, m, v = adam_step(logits, grads, m, v, step, lr=lr)
values = np.asarray(batch_value(logits))
print(
	f"repaired-measure ascent: best {values.max():.6f}, median {np.median(values):.4f} "
	f"({time.perf_counter() - start:,.0f} s; deterministic ceiling was 1.0)"
)

best = logits[int(np.argmax(values))]
p = np.asarray(jax.nn.sigmoid(best))
result = iit4.system_phi(System.from_state_by_node(jnp.asarray(p)), STATE3)
column0 = np.prod(1.0 - p, axis=1)
cause_code = int(
	sum(int(b) << i for i, b in enumerate(np.asarray(result.cause_effect_state.cause_state)))
)
p_c = column0[cause_code] / column0.sum()
print(
	f"anatomy: uncapped MIP = {float(result.signed_phi):.6f}, "
	f"cause surprisal = {-np.log2(p_c):.6f}, p_c = {p_c:.6f}"
)

repaired-measure ascent: best 1.391631, median 0.7504 (46 s; deterministic ceiling was 1.0)


anatomy: uncapped MIP = 1.391631, cause surprisal = 1.400205, p_c = 0.378875


## 5. The weighted-attractor search, retried (P5)


In [6]:
batch_phi4b = batch_phi_at_zero(N4)


def converges_to_zero(state_maps):
	pos = np.broadcast_to(np.arange(Q4), state_maps.shape).copy()
	for _ in range(Q4):
		pos = np.take_along_axis(state_maps, pos, axis=1)
	return (pos == 0).all(axis=1)


def maps_to_tables(state_maps):
	return ((state_maps[:, :, None] >> np.arange(N4)[None, None, :]) & 1).astype(np.float64)


rng3 = np.random.default_rng(SEED + 500)
# Seeds: heavy random funnels with in-degree(0) = 2 (the ladder's best attractor-
# compatible rung) plus e10-style trees.
population = np.zeros((256, Q4), dtype=np.int64)
for b in range(256):
	order = np.concatenate([[0], rng3.permutation(np.arange(1, Q4))])
	position = np.empty(Q4, dtype=np.int64)
	position[order] = np.arange(Q4)
	for s in range(1, Q4):
		population[b, s] = order[rng3.integers(0, position[s])]
values = np.asarray(batch_phi4b(jnp.asarray(maps_to_tables(population))))
start = time.perf_counter()
for generation in range(800):
	proposals = population.copy()
	rows = np.arange(256)
	for _ in range(1 + (generation % 3)):  # 1-3 simultaneous repointings
		targets = rng3.integers(1, Q4, size=256)
		proposals[rows, targets] = rng3.integers(0, Q4, size=256)
	ok = converges_to_zero(proposals)
	proposal_values = np.where(
		ok, np.asarray(batch_phi4b(jnp.asarray(maps_to_tables(proposals)))), -1.0
	)
	accept = proposal_values > values
	population[accept] = proposals[accept]
	values = np.where(accept, proposal_values, values)
print(
	f"retried attractor search: best = {values.max():.6f} "
	f"(e10 found 2.330; ladder bound n(n-1)/2 = 6.0) "
	f"({time.perf_counter() - start:,.0f} s)"
)
winner = population[int(np.argmax(values))]
print(f"winner map {winner.tolist()}, in-degree(0) = {int((winner == 0).sum())}")

retried attractor search: best = 3.500000 (e10 found 2.330; ladder bound n(n-1)/2 = 6.0) (41 s)
winner map [0, 15, 13, 10, 11, 11, 5, 13, 15, 15, 13, 13, 2, 9, 15, 0], in-degree(0) = 2


## Verdict

- **P3 confirmed — the attainment characterization is closed, exactly.** Adding the
  cause-side mirror at the unique preimage to the necessary condition yields
  **perfect set equality with the 3,591 capacity attainers: XOR = 0** over all
  16,777,216 systems. The theorem (at n = 3, by enumeration): φ_s(2023) = n(n−1)
  **iff** (i) the analyzed state has a unique preimage u*, (ii) each unit's output
  from s is realized by exactly one cross-input configuration (effect minterm), and
  (iii) each unit's output is realized exactly once among prior states agreeing
  with u* on that unit's self-input (cause minterm at the preimage). Forward and
  backward saturation, nothing else. e09's open sufficiency gap is closed;
  `maximum-theorem.md` and `metatheorems.md` upgraded.
- **P1/P2 invalidated by the instrument — and the instrument was caught by its own
  anchors.** The conditioned search reports g̃(1) = 4.245 at n = 4, but all-OR has
  in-degree 1 and φ = 12.000: the search undershoots the known value by 3×, so
  none of the n = 4 rung estimates bound anything, and the suppression
  measurements at those false winners are meaningless. Minterm-saturated systems
  are too rare and too isolated (the e01 gap, again) for random-plus-single-bit
  hill climbing to find. The n = 4 ladder needs *constructive* seeding — build
  candidate rung winners from the (now exact) characterization conditions — and is
  requeued. The registered ladder law remains untested above n = 3.
- **P4 confirmed — the repaired measure rewards stochasticity above all
  determinism.** Ascent on min(φ_2023, −log₂ p_c) reaches **1.3916** (registered
  (1, 3]), against its deterministic ceiling of 1.0. The optimum's anatomy repeats
  the n = 2 2026 pattern: the uncapped MIP equals φ exactly with the cause
  surprisal 9×10⁻³ above (p_c = 0.379) — a near-triple-point. The repaired
  measure's exact stochastic ceiling joins the n = 2 closed form in the
  derivation queue (same MIP-surprisal crossing family).
- **P5 refuted (but improved).** The retried attractor search reached 3.500
  (in-degree 2, up from e10's 2.330) — still below the registered 4.0 and the
  ladder bound n(n−1)/2 = 6.0. The n = 4 weighted-deterministic maximum now sits
  in [3.5, 6.0]; whether 6.0 is attainable jointly with global attraction remains
  the question.

**Standings.** The capacity story of the 2023 formalism is now complete at n = 3:
value theorem (n(n−1)), attainment characterization (exact iff: double minterm
saturation), and both are enumeration-proven. The n = 4 ladder is the main
casualty of honest instrumentation — requeued with constructive seeds — and the
repaired measure has grown its own theory question (a stochastic ceiling at a
MIP-surprisal crossing, alongside the n = 2 triple point).